# Übung 1: Die Daten der Druckerfarm verstehen
**Internet of Production – KI, Woche 1** · ca. 55 min · Arbeit in Zweiergruppen

**Szenario:** Drei 3D-Drucker fertigen Teile eines Didaktikbaukastens. Jeder Drucker protokolliert seinen Zustand **jede Minute**; Störungen werden separat protokolliert.

**Ziel:** Laden Sie die Daten eines Druckers, bereinigen Sie sie und finden Sie heraus, was an ihnen besonders ist. Am Ende suchen Sie ein **Warnzeichen vor einer Störung**: Es ist der Ausgangspunkt von Übung 2. Jedes `___` ist eine Lücke, die Sie füllen. Die Prüffragen beantworten Sie in den Textzellen.

| Datei | Inhalt |
|---|---|
| `Printer_0/1/2.csv` | Zustand, Temperaturen, Filamentsensor, Energiezähler (je Minute) |
| `Printer_0/1/2_error.csv` | Zeitpunkt und Art jeder Störung |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('../../00_Data/printfarm')   # folder with the CSV files
PRINTER = 1                               # start with printer 1

## Schritt 1: Ein erster Blick (naiv)
Führen Sie die Zelle aus. Was stimmt nicht?

In [ ]:
raw = pd.read_csv(DATA / f'Printer_{PRINTER}.csv')
print(raw.shape)
raw.head()

## Schritt 2: Datei richtig laden
Öffnen Sie die Datei in einem Texteditor (erste Zeilen). Welches **Trennzeichen** und welches **Dezimalzeichen** werden verwendet? Siehe `pd.read_csv(..., sep=..., decimal=...)`.

In [ ]:
df = pd.read_csv(DATA / f'Printer_{PRINTER}.csv', sep=___, decimal=___)
print(df.shape)
print(df.dtypes)
df.head()

## Schritt 3: Sind die Dateien vollständig und regelmäßig?
Prüfen Sie **alle drei Drucker**: Zahl der Zeilen, Zahl der Zeilen mit Daten (`dropna(how='all')`), Zeitraum und ob die Zeitstempel genau eine Minute auseinanderliegen (Lücken).

In [ ]:
for k in (0, 1, 2):
    d = pd.read_csv(DATA / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False)
    t = pd.to_datetime(d['Timestamp'].dropna(), format='%d.%m.%Y %H:%M')
    gaps = int((t.diff().dropna() != pd.Timedelta(minutes=___)).sum())
    print(f'printer {k}: rows {len(d):7d}, with data {len(d.___(how="all")):7d}, {t.min()} -> {t.max()}, gaps: {gaps}')

**Ihre Beobachtung:**

*Ihre Antwort:*

## Schritt 4: Bereinigen und Zeitstempel umwandeln
Entfernen Sie leere Zeilen, wandeln Sie `Timestamp` (Format `Tag.Monat.Jahr Stunde:Minute`) in ein Datum um und prüfen Sie den Abtastabstand.

In [ ]:
df = df.___(how='all').copy()
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format=___)
df = df.set_index('Timestamp')
print(df.index.min(), '->', df.index.max())
print(df.index.to_series().diff().value_counts().head())

## Schritt 5: Der Energiezähler
`Total_Power` sieht aus wie ein Messwert. Stellen Sie ihn über die Zeit dar. Was für eine Größe ist das? Leiten Sie die **Energie je Minute** ab und vergleichen Sie die Zustände.

In [ ]:
df['Total_Power'].plot(figsize=(10, 3), title='Total_Power')
plt.show()

In [ ]:
df['Energy'] = df['Total_Power'].___()
print(df.groupby(___)['Energy'].mean().round(5))
print('negative steps:', (df['Energy'] < 0).sum())

**Was ist `Total_Power`, und welcher Zustand verbraucht am meisten?**

*Ihre Antwort:*

## Schritt 6: Zustände
Wie verteilen sich die Minuten auf die Zustände?

In [ ]:
share = df['State'].___(normalize=True).mul(100).round(1)
print(share)
share.plot.barh(title='Share of minutes per state (%)')
plt.show()

## Schritt 7: Störungen
Laden Sie die Störungsdatei des Druckers und zählen Sie die Störungen je Art. Betrachten Sie dann die Minutendaten **um die erste Extruder-Blockade**: In welchem Zustand ist der Drucker, und was passiert mit der Extrudertemperatur?

In [ ]:
err = pd.read_csv(DATA / f'Printer_{PRINTER}_error.csv', sep=';').dropna(how='all')
err['Timestamp'] = pd.to_datetime(err['Timestamp'], format='%d.%m.%Y %H:%M')
print(err['Error'].___())
first = err.loc[err['Error'] == 'Extruder blocked', 'Timestamp'].iloc[0]
print('first extruder blockage:', first)
df.loc[first - pd.Timedelta(minutes=3): first + pd.Timedelta(minutes=3), ['State', 'Temp_Extruder', 'Temp_Bed']]

Wie lange **bleibt** ein Drucker im Zustand `error`? Zählen Sie die zusammenhängenden Episoden und betrachten Sie ihre Länge in Minuten.

In [ ]:
episode = (df['State'] != df['State'].shift()).cumsum()
lengths = df.groupby(episode).agg(state=('State', 'first'), minutes=('State', 'size'))
e = lengths[lengths['state'] == ___]['minutes']
print('episodes in state error:', len(e))
print(e.describe().round(1))

## Schritt 8: Gibt es ein Warnzeichen vor einer Extruder-Blockade?
Eine einzelne Minute des Energieverbrauchs streut stark (der Zähler hat eine Auflösung von 0,1 Wh). Bilden Sie daher **Mittelwerte**: Berechnen Sie für jede der 12 Stunden vor einer Extruder-Blockade den mittleren Energieverbrauch je Minute **im Zustand printing** über alle Blockaden dieses Druckers und vergleichen Sie ihn mit dem normalen Druckbetrieb.

In [ ]:
printing = df[df['State'] == 'printing']
normal = printing['Energy'].___() * 1000        # Wh per minute (if Total_Power is in kWh)
blocked = err.loc[err['Error'] == 'Extruder blocked', 'Timestamp']

rows = []
for t in blocked:
    for h in range(12, 0, -1):                   # hour bins: 12 h ... 1 h before the blockage
        w = printing.loc[t - pd.Timedelta(hours=h): t - pd.Timedelta(hours=h - 1, minutes=1), 'Energy']
        if len(w) >= 30:
            rows.append((h, w.___() * 1000))
profile = pd.DataFrame(rows, columns=['hours_before', 'Wh_per_min']).groupby('hours_before')['Wh_per_min'].mean()

ax = profile.plot(marker='o', figsize=(9, 3.5), color='#003A79', label='before an extruder blockage')
ax.axhline(normal, color='grey', ls='--', label='normal printing')
ax.invert_xaxis(); ax.set_xlabel('hours before the blockage'); ax.set_ylabel('energy per minute (Wh)'); ax.legend(); plt.show()

Zeigen die anderen Störungsarten dasselbe? Vergleichen Sie den mittleren Energieverbrauch in den 6 Stunden vor jeder Störungsart mit dem normalen Druckbetrieb.

In [ ]:
for typ, ts in err.groupby('Error')['Timestamp']:
    vals = []
    for t in ts:
        w = printing.loc[t - pd.Timedelta(hours=6): t - pd.Timedelta(minutes=1), 'Energy']
        if len(w) >= 30:
            vals.append(w.mean() * 1000)
    print(f'{typ:24s} n = {len(vals):2d}   energy in the 6 h before: {np.___(vals):.3f} Wh/min   (normal: {normal:.3f})')

**Was sehen Sie? Warum brauchen wir den Mittelwert, und bei welchen Störungsarten zeigt es sich?**

*Ihre Antwort:*

## Schritt 9: Wiederverwendbar machen
Fassen Sie Ihre Bereinigungsschritte in einer Funktion zusammen. Wir nutzen sie in Woche 3 wieder.

In [ ]:
def load_printer(k, data=DATA):
    """Minute data of printer k: clean, indexed by time, with the column Energy (per minute)."""
    d = pd.read_csv(data / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False)
    # TODO: drop empty rows, parse Timestamp, use it as index, add the column Energy
    ___
    return d

def load_errors(k, data=DATA):
    # TODO: read the error file, drop empty rows, parse Timestamp
    ___

p2 = load_printer(2)
print(p2.shape, p2.index.min(), p2.index.max())

## Ihre Erkenntnisse (drei bis vier Sätze)
Was ist Ihnen an den Daten aufgefallen? Was wäre schiefgegangen, hätten Sie sie ungeprüft verwendet?

*Ihre Antwort:*